# Khmer OCR — canvas expansionMakes the text look **smaller** by growing the image around it. The Khmer isnever resized; only the white space changes.The recognizer normalises every input to height 64, so:```text height the model sees  =  64 / expansion```| expansion | text seen | word fills ||---|---|---|| 1.5× | 43px | 67% || 2.0× | 32px | 50% || 3.0× | 21px | 33% || 4.0× | 16px | 25% |Run the cells in order.

In [ ]:
# 1 — setup!pip -q install -U pillow pandasimport io, os, random, shutil, zipfilefrom pathlib import Pathimport pandas as pdfrom PIL import Image, ImageFilterfrom google.colab import drivedrive.mount('/content/drive')SEED = 42random.seed(SEED)DRIVE = Path('/content/drive/MyDrive/KhmerOCR/datasets')WORK  = Path('/content/khmer_data')WORK.mkdir(parents=True, exist_ok=True)print('drive :', DRIVE, '->', 'OK' if DRIVE.exists() else 'NOT FOUND')for f in sorted(DRIVE.glob('*.zip')):    print(f'   {f.name:32s} {f.stat().st_size/1e6:8.1f} MB')

In [ ]:
# 2 — unzip the source dataset out of Drive onto fast local diskSRC = WORK / 'existing_dataset'if not SRC.exists():    print('unzipping existing_dataset.zip ...')    with zipfile.ZipFile(DRIVE / 'existing_dataset.zip') as z:        z.extractall(SRC)df = pd.read_csv(SRC / 'labels.csv')print(f'{len(df)} rows, {len(list((SRC/"images").iterdir()))} images')print(df.head(3).to_string(index=False))

## Settings`EXPANSIONS` is the list of canvas multipliers — one is picked at random perimage. `KEEP_CLEAN` is the fraction left untouched so the model does not forgetnormal-sized text.**Measured on the current model**, for reference:```expansion  1.5×   2.0×    3.2×    4.0×    6.0×+CER         ~30%  67.9%   98.2%   99.5%   ~100%```Above 4× the model reads nothing at all, which gives no useful training signal.Start in the 1.5–4× range.

In [ ]:
# 3 — settingsPREP_H     = 64EXPANSIONS = [1.5, 2.0, 3.0, 4.0]KEEP_CLEAN = 0.30SHAPE      = 'portrait'     # 'portrait' | 'balanced' | 'wide'OUT_NAME   = 'dataset_09_expanded'print('effective text height after prep():')for e in EXPANSIONS:    print(f'   {e:4.1f}x  ->  {PREP_H/e:5.1f}px   (word fills {100/e:.0f}% of height)')

In [ ]:
# 4 — the expansion functiondef expand(img, expansion, shape=SHAPE, rng=random, jitter=0.25, blur_p=0.25):    """Grow the canvas around the text. The text itself is never resized."""    w, h  = img.size    new_h = max(h + 2, int(round(h * expansion)))    if shape == 'portrait':        new_w = max(w + 2, int(round(new_h * rng.uniform(0.5, 0.9))))    elif shape == 'wide':        new_w = int(round(w * expansion))    else:        new_w = max(w + 2, int(round(w * expansion)))    canvas = Image.new('L', (new_w, new_h), 255)    fx, fy = new_w - w, new_h - h    x = max(0, min(fx, int(fx // 2 + rng.uniform(-jitter, jitter) * fx)))    y = max(0, min(fy, int(fy // 2 + rng.uniform(-jitter, jitter) * fy)))    canvas.paste(img, (x, y))    if rng.random() < blur_p:        canvas = canvas.filter(ImageFilter.GaussianBlur(rng.uniform(0.2, 0.6)))    return canvas, (x, y, x + w, y + h), PREP_H / expansion

In [ ]:
# 5 — preview before committing to a full runfrom IPython.display import displaysample = df.sample(3, random_state=SEED)for _, r in sample.iterrows():    before = Image.open(SRC / 'images' / r['filename']).convert('L')    print(f"\n{r['text']}")    print(f"  BEFORE  {before.size[0]}x{before.size[1]}  aspect {before.size[0]/before.size[1]:.2f}")    display(before)    for e in EXPANSIONS:        after, box, eff = expand(before, e, rng=random.Random(SEED))        print(f"  {e:.1f}x -> {after.size[0]}x{after.size[1]}  "              f"aspect {after.size[0]/after.size[1]:.2f}  text ~{eff:.0f}px")        display(after.resize((min(500, after.size[0]),                              int(after.size[1] * min(500, after.size[0]) / after.size[0]))))

In [ ]:
# 6 — generate the full datasetOUT = WORK / OUT_NAME(OUT / 'images').mkdir(parents=True, exist_ok=True)rng = random.Random(SEED)labels = []for i, r in df.iterrows():    try:        img = Image.open(SRC / 'images' / r['filename']).convert('L')    except Exception:        continue    name = f'expanded_{i:07d}.jpg'    rec  = {'filename': name, 'text': r['text']}    if rng.random() < KEEP_CLEAN:        img.save(OUT / 'images' / name, quality=92)        rec.update({'expansion': 1.0, 'eff_text_px': PREP_H,                    'aspect': round(img.size[0] / img.size[1], 2)})    else:        e = rng.choice(EXPANSIONS)        canvas, box, eff = expand(img, e, rng=rng)        canvas.save(OUT / 'images' / name, quality=rng.randint(75, 95))        rec.update({'expansion': e, 'eff_text_px': round(eff, 1),                    'x0': box[0], 'y0': box[1], 'x1': box[2], 'y1': box[3],                    'canvas_w': canvas.size[0], 'canvas_h': canvas.size[1],                    'aspect': round(canvas.size[0] / canvas.size[1], 2)})    labels.append(rec)    if (i + 1) % 2000 == 0:        print(f'  {i+1}/{len(df)}', flush=True)out_df = pd.DataFrame(labels)out_df.to_csv(OUT / 'labels.csv', index=False)print(f'\n{len(out_df)} samples -> {OUT}')print(out_df.groupby('expansion').size().to_string())

## ValidationSix checks. The one that matters is the last: crop each image by its saved boxand confirm the crop contains the text. If the boxes were wrong or pointed atblank paper, the crops would be unreadable.

In [ ]:
# 7 — validateimport numpy as npfails, blanks, bad_box = [], 0, 0lens = []for _, r in out_df.iterrows():    p = OUT / 'images' / r['filename']    try:        im = Image.open(p).convert('L')    except Exception:        fails.append(r['filename']); continue    if not isinstance(r['text'], str) or not r['text'].strip():        fails.append(r['filename'])    lens.append(len(str(r['text'])))    if np.asarray(im, dtype=np.uint8).std() < 2.0:        blanks += 1    if r['expansion'] > 1:        if not (0 <= r['x0'] < r['x1'] <= r['canvas_w']                and 0 <= r['y0'] < r['y1'] <= r['canvas_h']):            bad_box += 1print('1 structure  ', 'OK' if not fails  else f'{len(fails)} bad')print('2 length     ', f'min {min(lens)} / mean {sum(lens)/len(lens):.0f} / max {max(lens)}',      '  OK' if max(lens) <= 60 else '  SOME OVER 60 (CTC limit)')print('3 content    ', 'OK' if not blanks else f'{blanks} near-blank images')print('4 boxes      ', 'OK' if not bad_box else f'{bad_box} invalid')exp = out_df[out_df.expansion > 1]print(f'5 aspect      min {exp.aspect.min():.2f} / median {exp.aspect.median():.2f} / max {exp.aspect.max():.2f}')print(f'6 crop check  showing 3 crops -- each should be readable Khmer:')for _, r in exp.sample(3, random_state=SEED).iterrows():    im   = Image.open(OUT / 'images' / r['filename']).convert('L')    crop = im.crop((r['x0'], r['y0'], r['x1'], r['y1']))    print(f"   {r['text']}   ({r['expansion']}x, text ~{r['eff_text_px']}px)")    display(crop)

In [ ]:
# 8 — zip back to Driveprint('zipping ...')shutil.make_archive(str(DRIVE / OUT_NAME), 'zip', root_dir=OUT)z = DRIVE / f'{OUT_NAME}.zip'print(f'done: {z}  ({z.stat().st_size/1e6:.1f} MB)')!ls -lh "/content/drive/MyDrive/KhmerOCR/datasets"